# Patronus

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

READWISE_TOKEN = os.getenv("READWISE_TOKEN")

XvSKyS975HbqD43CIpE4Y9YVohr4aUmhELo4pN3g7OnS6jBk6J


In [3]:
from typing import List, Dict
import os
from enum import Enum

import feedparser
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY: str | None = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPENAI_API_KEY in environment")
client = OpenAI(api_key=OPENAI_API_KEY)

class Bucket(str, Enum):
    REJECT = "reject"
    TECHNICAL_AI_ML = "technical_ai_ml"
    AI_SAFETY_BUSINESS = "ai_safety_business"
    PHILOSOPHY_CONSCIOUSNESS = "philosophy_consciousness"
    POLITICS_CULTURE = "politics_culture"
    SPAIN = "spain"
    CHINA = "china"
    RANDOM_CURIOSITIES = "random_curiosities"

class ArticleClassification(BaseModel):
    bucket: Bucket = Field(..., description="Selected bucket for the article")
    reason: str = Field(..., description="Concise justification for the classification")

feeds_path: str = "/Users/dani/code/patronus/feeds"
with open(feeds_path, "r") as f:
    feed_urls: List[str] = [line.strip() for line in f if line.strip()]



In [6]:
from typing import List, Dict, Optional, TypedDict
import os
from enum import Enum
from datetime import datetime
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY: Optional[str] = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPENAI_API_KEY in environment")
client: OpenAI = OpenAI(api_key=OPENAI_API_KEY)

class Bucket(str, Enum):
    REJECT = "reject"
    TECHNICAL_AI_ML = "technical_ai_ml"
    AI_SAFETY_BUSINESS = "ai_safety_business"
    PHILOSOPHY_CONSCIOUSNESS = "philosophy_consciousness"
    POLITICS_CULTURE = "politics_culture"
    SPAIN = "spain"
    CHINA = "china"
    RANDOM_CURIOSITIES = "random_curiosities"

class ArticleClassification(BaseModel):
    bucket: Bucket
    reason: str

class Article(TypedDict):
    title: str
    link: str
    content: str
    published: Optional[datetime]



In [13]:
import feedparser
from time import mktime
from trafilatura import fetch_url, extract

feeds_path: str = "/Users/dani/code/patronus/feeds"
with open(feeds_path, "r") as f:
    feed_urls: List[str] = [line.strip() for line in f if line.strip()]

meta_entries: List[Dict] = []
for url in feed_urls:
    parsed = feedparser.parse(url)
    feed_meta: List[Dict] = []
    for e in parsed.entries:
        published_dt: Optional[datetime] = None
        if hasattr(e, "published_parsed") and e.published_parsed:
            published_dt = datetime.fromtimestamp(mktime(e.published_parsed))
        elif hasattr(e, "updated_parsed") and e.updated_parsed:
            published_dt = datetime.fromtimestamp(mktime(e.updated_parsed))
        feed_meta.append({
            "title": getattr(e, "title", ""),
            "link": getattr(e, "link", ""),
            "summary": getattr(e, "summary", getattr(e, "description", "")),
            "published": published_dt,
        })
    feed_meta_sorted: List[Dict] = sorted(
        feed_meta,
        key=lambda x: (x["published"] or datetime.min),
        reverse=True,
    )
    meta_entries.extend(feed_meta_sorted[:10])

top_meta: List[Dict] = sorted(
    meta_entries,
    key=lambda x: (x["published"] or datetime.min),
    reverse=True,
)[:50]

articles: List[Article] = []
for m in top_meta:
    link: str = m["link"]
    content_text: str = ""
    html: Optional[str] = fetch_url(link) if link else None
    if html:
        extracted: Optional[str] = extract(html, include_comments=False, include_tables=False)
        if extracted:
            content_text = extracted
    if not content_text:
        content_text = m.get("summary", "")
    articles.append({
        "title": m["title"],
        "link": link,
        "content": content_text,
        "published": m["published"],
    })
len(articles)


50

In [9]:
profile_path: str = "/Users/dani/code/patronus/Profile.md"
with open(profile_path, "r") as pf:
    dani_profile: str = pf.read()

system_preamble: str = (
    "You are an assistant tasked with selecting content from an RSS feed that is relevant to Dani. "
    "Classify each article into one of the following buckets based on Dani's profile: "
    "REJECT, TECHNICAL_AI_ML, AI_SAFETY_BUSINESS, PHILOSOPHY_CONSCIOUSNESS, POLITICS_CULTURE, SPAIN, CHINA, RANDOM_CURIOSITIES."
)


In [10]:
from pydantic import BaseModel

class ArticleClassification(BaseModel):
    bucket: Bucket
    reason: str

from typing import Tuple

def classify_article(article: Article) -> ArticleClassification:
    user_prompt: str = (
        f"Profile about Dani (verbatim):\n\n{dani_profile}\n\n"
        f"Article to classify:\nTitle: {article['title']}\nLink: {article['link']}\nContent (truncated to 4000 chars):\n{article['content'][:4000]}\n\n"
        f"Return only valid JSON matching this schema (no extra keys):\n{ArticleClassification.model_json_schema()}"
    )
    comp = client.chat.completions.parse(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": system_preamble},
            {"role": "user", "content": user_prompt},
        ],
        response_format=ArticleClassification,
    )
    parsed: ArticleClassification | None = comp.choices[0].message.parsed
    if parsed is None:
        content: str = comp.choices[0].message.content or "{}"
        parsed = ArticleClassification.model_validate_json(content)
    assert parsed is not None
    return parsed



In [14]:
from collections import defaultdict, Counter

buckets: Dict[Bucket, List[Article]] = defaultdict(list)
classifications: List[ArticleClassification] = []

for art in articles:
    classification = classify_article(art)
    classifications.append(classification)
    buckets[classification.bucket].append(art)

counts = Counter([c.bucket.value for c in classifications])
print("Summary Statistics:")
for b in [bucket.value for bucket in Bucket]:
    print(f"{b}: {counts.get(b, 0)}")

# Inspect a sample per bucket (titles only)
{b.value: [a['title'] for a in buckets[b]][:3] for b in Bucket}


Summary Statistics:
reject: 7
technical_ai_ml: 15
ai_safety_business: 7
philosophy_consciousness: 2
politics_culture: 11
spain: 0
china: 1
random_curiosities: 7


{'reject': ['[Action required] Your RSS.app Trial has Expired.',
  'Quoting Matt Garman',
  'Quoting u/AssafMalkiIL'],
 'technical_ai_ml': ['too many model context protocol servers and LLM allocations on the dance floor',
  'DeepSeek v3.1 Is Not Having a Moment',
  'Quoting potatolicious'],
 'ai_safety_business': ['AI #130: Talking Past The Sale',
  'AI Companion Conditions',
  'GPT-5: The Reverse DeepSeek Moment'],
 'philosophy_consciousness': ['Quoting Mustafa Suleyman',
  'My Responses To Three Concerns From The Embryo Selection Post'],
 'politics_culture': ['Chartbook 405 Bulldozing Gaza: (Thanatocene mini-series #4)',
  "Top Links 837 American globalization & Armington elasticity. The world's ten fastest growing cities & toasting to the success of our hopeless cause.",
  'Top Links 836 Privilège exorbitant. The $6.3 Trillion India-China Stock Gap. After Kant and the "Devil\'s Disciple".'],
 'spain': [],
 'china': ["Nongfu Spring and China's Bottled Water Wars"],
 'random_curiositi

In [15]:
import os
from typing import Tuple
from feedgen.feed import FeedGenerator
from google.cloud import storage

from dotenv import load_dotenv

load_dotenv()


GCS_BUCKET_NAME: str = os.getenv("GCS_BUCKET_NAME", "")
GCS_PREFIX: str = os.getenv("GCS_PREFIX", "patronus/feeds/")
if not GCS_BUCKET_NAME:
    raise RuntimeError("Missing GCS_BUCKET_NAME in environment")

storage_client = storage.Client()
bucket = storage_client.bucket(GCS_BUCKET_NAME)

def build_rss_xml(bucket_key: Bucket, items: List[Article]) -> str:
    fg = FeedGenerator()
    fg.title(f"Patronus: {bucket_key.value}")
    fg.link(href="https://example.com", rel="alternate")
    fg.description(f"Filtered feed for {bucket_key.value}")
    for it in items:
        fe = fg.add_entry()
        fe.title(it["title"]) 
        fe.link(href=it["link"])
        if it.get("published"):
            fe.published(it["published"]) 
        fe.description(it.get("content", ""))
    return fg.rss_str(pretty=True).decode("utf-8")

public_urls: Dict[str, str] = {}
for b in Bucket:
    key = f"{GCS_PREFIX}{b.value}.xml"
    xml_data: str = build_rss_xml(b, buckets.get(b, []))
    blob = bucket.blob(key)
    blob.upload_from_string(xml_data, content_type="application/rss+xml")
    blob.acl.reload()
    blob.make_public()
    public_urls[b.value] = blob.public_url

public_urls


/Users/dani/code/patronus/.venv/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/dani/code/patronus/.venv/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


ValueError: Datetime object has no timezone info

In [16]:
from typing import Dict, List, Optional
from datetime import datetime, timezone
from feedgen.feed import FeedGenerator
from google.cloud import storage


def _sanitize_text(text: str) -> str:
    if not isinstance(text, str):
        try:
            text = str(text)
        except Exception:
            return ""
    return "".join(ch for ch in text if (ch >= " " or ch in "\n\r\t"))


def _ensure_tz(dt: Optional[datetime]) -> Optional[datetime]:
    if dt is None:
        return None
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt


def build_rss_xml(bucket_key: Bucket, items: List[Article]) -> str:
    fg = FeedGenerator()
    fg.title(f"Patronus: {bucket_key.value}")
    fg.link(href="https://example.com", rel="alternate")
    fg.description(f"Filtered feed for {bucket_key.value}")
    for it in items:
        title_val = _sanitize_text(it.get("title", "")) or "Untitled"
        content_val = _sanitize_text(it.get("content", ""))
        link_val = it.get("link", "")
        fe = fg.add_entry()
        fe.title(title_val)
        if link_val:
            fe.link(href=link_val)
        pub_dt = _ensure_tz(it.get("published"))
        if pub_dt is not None:
            fe.published(pub_dt)
        fe.description(content_val)
    return fg.rss_str(pretty=True).decode("utf-8")


GCS_BUCKET_NAME: str = os.getenv("GCS_BUCKET_NAME", "")
GCS_PREFIX: str = os.getenv("GCS_PREFIX", "patronus/feeds/")
if not GCS_BUCKET_NAME:
    raise RuntimeError("Missing GCS_BUCKET_NAME in environment")

storage_client = storage.Client()
bucket_client = storage_client.bucket(GCS_BUCKET_NAME)

public_urls: Dict[str, str] = {}
for b in Bucket:
    key = f"{GCS_PREFIX}{b.value}.xml"
    xml_data: str = build_rss_xml(b, buckets.get(b, []))
    blob = bucket_client.blob(key)
    blob.upload_from_string(xml_data, content_type="application/rss+xml")
    try:
        blob.make_public()
    except Exception:
        pass
    public_urls[b.value] = blob.public_url

public_urls


/Users/dani/code/patronus/.venv/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/Users/dani/code/patronus/.venv/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


{'reject': 'https://storage.googleapis.com/danibalcellsrss//reject.xml',
 'technical_ai_ml': 'https://storage.googleapis.com/danibalcellsrss//technical_ai_ml.xml',
 'ai_safety_business': 'https://storage.googleapis.com/danibalcellsrss//ai_safety_business.xml',
 'philosophy_consciousness': 'https://storage.googleapis.com/danibalcellsrss//philosophy_consciousness.xml',
 'politics_culture': 'https://storage.googleapis.com/danibalcellsrss//politics_culture.xml',
 'spain': 'https://storage.googleapis.com/danibalcellsrss//spain.xml',
 'china': 'https://storage.googleapis.com/danibalcellsrss//china.xml',
 'random_curiosities': 'https://storage.googleapis.com/danibalcellsrss//random_curiosities.xml'}